### Simple Gen AI App using Langchain

In [10]:
import os
from dotenv import load_dotenv
load_dotenv()

OPENROUTER_API_KEY=os.getenv("OPENROUTER_API_KEY") or ""
os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY") or ""
os.environ["LANGSMITH_TRACING"]=os.getenv("LANGSMITH_TRACING") or ""
os.environ["LANGSMITH_PROJECT"]=os.getenv("LANGSMITH_PROJECT") or ""

#### Workflow
  
`Load Data -> Docs -> Divide docs into chunk -> Convert to Vectors -> Vector Embeddings -> Vector Store DB -> Query the DB -> Retrieval Chaining -> Response from LLM`

#### Data Ingestion (Load data)
From the website we need to scrape the data and ingest it into a vector database. 

In [65]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://docs.langchain.com/langsmith/billing")
loader

In [66]:
docs = loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/billing', 'title': 'Manage billing in your account - Docs by LangChain', 'language': 'en'}, page_content='Manage billing in your account - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationBilling & usageManage billing in your accountGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage billingGranular usageManage organizations using the APIToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusOn this pageSet up billing for your accountDeveloper plan: set up billing on your personal organizationPlus plan: set up billing on a shared organizationTrack contract usage (Enterprise)V

#### Data transformation (Splitting data into chunks)

In [67]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
docs = text_splitter.split_documents(docs)

In [68]:
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/billing', 'title': 'Manage billing in your account - Docs by LangChain', 'language': 'en'}, page_content='Manage billing in your account - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationBilling & usageManage billing in your accountGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage billingGranular usageManage organizations using the APIToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusOn this pageSet up billing for your accountDeveloper plan: set up billing on your personal organizationPlus plan: set up billing on a shared organizationTrack contract usage (Enterprise)V

#### Vector Embedding & Vector Store DB

In [69]:
from langchain_openai import OpenAIEmbeddings
from pydantic import SecretStr

embeddings = OpenAIEmbeddings(
    api_key=SecretStr(OPENROUTER_API_KEY),
    base_url="https://openrouter.ai/api/v1"
)


In [70]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(docs, embeddings)
db

#### Query Vector Store DB

In [72]:
query = "To set up billing for your LangSmith organization"
res = db.similarity_search(query)
print(res[0].page_content)

Set up billing for your account: Complete the billing setup process for Developer and Plus plans, including special instructions for legacy accounts.
Track contract usage (Enterprise): Monitor your prepaid commitment consumption and usage patterns.
Update your information: Modify invoice email addresses, business information, and tax IDs for your organization.
Enforce spend limits: Learn how to manage your spend through usage limits and data retention.

​Set up billing for your account
Before using this guide, note the following:
If you are interested in the Enterprise plan, please contact sales. This guide is only for our self-serve billing plans.

To set up billing for your LangSmith organization, navigate to the Billing and Usage page under Settings. Depending on your organization’s settings, there are different setup guides:

Developer plan
Plus plan


#### Retrieval Chain
Retrieves relevant documents from the vector store and feeds them into the LLM to generate a response.  

##### Document Chain 
It takes in documents and processes them in some way, such as summarizing them or extracting information from them.

In [76]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model = "openai/gpt-oss-120b:free",
    api_key = SecretStr(OPENROUTER_API_KEY),
    base_url = "https://openrouter.ai/api/v1"
)

In [95]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
        Answer the following question based only on the provided context:
        <context>
        {context}
        </context>
        
        Question: {input}
    """
)

doc_chain = create_stuff_documents_chain(llm, prompt)
doc_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\n        Answer the following question based only on the provided context:\n        <context>\n        {context}\n        </context>\n\n        Question: {input}\n    '), additional_kwargs={})])
| ChatOpenAI(profile={}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000025AC1FEC050>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000025AC1FEC2D0>, root_client=<openai.OpenAI object at 0x0000025ABDECF110>, root_async_client=<openai.AsyncOpenAI object at 0x0000025ABDECED50>, model_name='openai/gpt-o

In [96]:
from langchain_core.documents import Document

doc_chain.invoke({
    "input": "LangSmith has two trace tiers: base traces and extended traces",
    "context": [Document(page_content = "LangSmith has two trace tiers: base traces and extended traces. Base traces have the base retention and are short-lived (14 days), while extended traces have extended retention and are long-lived (400 days by default, customizable for Enterprise customers).")]
})

'Yes. LangSmith provides two trace tiers:\n\n- **Base traces** – use the standard (short‑term) retention of 14\u202fdays.  \n- **Extended traces** – use the longer‑term retention (400\u202fdays by default, with further customization available for Enterprise customers).'

However, we want the documents to first come from the retriever we just set up. That way, we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.

In [97]:
from langchain_classic.chains import create_retrieval_chain

retriever = db.as_retriever()
retrievel_chain = create_retrieval_chain(retriever, doc_chain)
retrievel_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000025ABC84D5B0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\n        Answer the following question based only on the provided context:\n        <context>\n        {context}\n        </contex

##### Get response from the LLM

In [100]:
response = retrievel_chain.invoke({
    "input": "LangSmith has two trace tiers: base traces and extended traces"
})
response['answer']

'Yes.\u202fLangSmith offers two trace tiers:\n\n* **Base traces** – the default tier with short‑term retention (14\u202fdays).  \n* **Extended traces** – a longer‑term tier with extended retention (400\u202fdays by default, and customizable for Enterprise customers).  \n\nExtended traces cost more than base traces, so choosing them as the default reduces the total number of traces you can generate in a billing period. The tier can be set as the default for all new traces, and you can further distribute limits between the two tiers with a slider when the default is set to Base. Additionally, certain automation rules can automatically upgrade a trace from Base to Extended retention.'

In [102]:
response['context']

[Document(id='54b04357-1ccb-476a-93f1-ea033e0f6f02', metadata={'source': 'https://docs.langchain.com/langsmith/billing', 'title': 'Manage billing in your account - Docs by LangChain', 'language': 'en'}, page_content='For organizations with multiple workspaces only: For simplicity, LangSmith incorporates the free traces into the cost calculation of the first workspace only. In actuality, the free traces can be “consumed” by any workspace. Therefore, although workspace-level spend limits are approximate for multi-workspace organizations, the organization-level spend limit is absolute.\n\u200bConfigure trace tier distribution\nLangSmith has two trace tiers: base traces and extended traces. Base traces have the base retention and are short-lived (14 days), while extended traces have extended retention and are long-lived (400 days by default, customizable for Enterprise customers). For more information, refer to the data retention conceptual docs.'),
 Document(id='5035fbb1-7ee6-48f8-bd18-15

In [113]:
contexts = response.get("context", [])

for idx, doc in enumerate(contexts, start=1):
    print(f"\n--- Context {idx} ---\n{doc.page_content}")


--- Context 1 ---
For organizations with multiple workspaces only: For simplicity, LangSmith incorporates the free traces into the cost calculation of the first workspace only. In actuality, the free traces can be “consumed” by any workspace. Therefore, although workspace-level spend limits are approximate for multi-workspace organizations, the organization-level spend limit is absolute.
​Configure trace tier distribution
LangSmith has two trace tiers: base traces and extended traces. Base traces have the base retention and are short-lived (14 days), while extended traces have extended retention and are long-lived (400 days by default, customizable for Enterprise customers). For more information, refer to the data retention conceptual docs.

--- Context 2 ---
Set the desired default trace tier by selecting an option below the Default data retention label. All traces will have this tier by default when they are registered. Note that because extended traces cost more than base traces, s